# MODELOS

IMPORTACION DE LIBRERIAS

In [2]:
# Bibliotecas estándar
import joblib
from collections import Counter

# Bibliotecas de análisis y modelado de datos
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score

# Librerias de modelos
#from statsmodels.tsa.statespace.sarimax import SARIMAX
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# Bibliotecas de visualización
from imblearn.over_sampling import SMOTE


CARGA DE DATOS

In [3]:
df = pd.read_csv(r'.\data\observations_full.csv')

df['date'] = pd.to_datetime(df['date'])

df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day

SVM

In [4]:
# Select features and target
features = ['year', 'month', 'day', 'precipitation', 'wind', 'humidity', 'estacion_id']
target = 'weather_id'

# Split data into train and test sets
X = df[features]
y = df[target]

# Handle class imbalance with SMOTE
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, random_state=42, stratify=y_res)

# Scale the data (using StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = SVC(probability=True, C=0.1, gamma='scale', kernel='rbf', class_weight='balanced')
model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

# Classification Report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# ROC AUC (for multi-class)
roc_auc = roc_auc_score(y_test, model.predict_proba(X_test_scaled), multi_class='ovr')
print(f"ROC AUC: {roc_auc}")

# Save the model using joblib
#joblib.dump(model, r'.\main\Modelos\SVM_weather_id.pkl')

Accuracy: 0.7854054624064115
Classification Report:
              precision    recall  f1-score   support

           1       0.82      0.86      0.84      1897
           2       0.79      0.82      0.81      1896
           3       0.75      0.44      0.56      1897
           4       0.68      0.84      0.75      1897
           5       0.89      0.96      0.92      1896

    accuracy                           0.79      9483
   macro avg       0.79      0.79      0.78      9483
weighted avg       0.79      0.79      0.78      9483

Confusion Matrix:
[[1631    2  116  148    0]
 [  15 1557  150  174    0]
 [ 255  263  835  360  184]
 [  96  149    6 1602   44]
 [   0    0    3   70 1823]]
ROC AUC: 0.9429371569285117


XGBoosting Classifier

In [6]:
# Seleccionar las características y el objetivo
features = ['year', 'month', 'day', 'precipitation', 'wind', 'humidity', 'estacion_id']
target = 'weather_id'

# Dividir los datos en características (X) y objetivo (y)
X = df[features]
y = df[target]

# Ajustar las etiquetas para que comiencen desde 0
y_adjusted = y - 1  # Ahora las clases estarán en el rango [0, 1, 2, 3, 4]

# Mostrar la distribución original de las clases
print("Distribución original de las clases:", Counter(y_adjusted))

# Crear una instancia de SMOTE
smote = SMOTE(random_state=42, k_neighbors=10)
X_resampled, y_resampled = smote.fit_resample(X, y_adjusted)

# Mostrar la nueva distribución de las clases
print("Distribución de clases después de SMOTE:", Counter(y_resampled))

# Dividir los datos balanceados en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

# Crear un modelo de XGBoost
model = XGBClassifier(random_state=42, n_estimators=300, max_depth=30)

# Ajustar el modelo con los datos de entrenamiento balanceados
model.fit(X_train, y_train)

# Realizar predicciones sobre el conjunto de prueba
y_pred = model.predict(X_test)

# Reconvertir las predicciones y las etiquetas originales
y_pred = y_pred + 1  # Volver a [1, 2, 3, 4, 5]
y_test = y_test + 1  # Volver a [1, 2, 3, 4, 5]

# Calcular la exactitud (accuracy)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

# Evaluar el rendimiento del modelo
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred))

# Mostrar la matriz de confusión
print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

# Guardar el modelo entrenado usando joblib
joblib.dump(model, r'..\main\Modelos\XGB_weather_id.pkl')

Distribución original de las clases: Counter({1: 9483, 0: 9024, 2: 5959, 3: 429, 4: 105})
Distribución de clases después de SMOTE: Counter({0: 9483, 2: 9483, 1: 9483, 3: 9483, 4: 9483})
Accuracy: 0.9244964673626489

Reporte de clasificación:
              precision    recall  f1-score   support

           1       0.89      0.97      0.93      1897
           2       0.89      0.96      0.92      1910
           3       0.93      0.71      0.80      1861
           4       0.93      0.98      0.95      1892
           5       0.99      1.00      0.99      1923

    accuracy                           0.92      9483
   macro avg       0.93      0.92      0.92      9483
weighted avg       0.93      0.92      0.92      9483


Matriz de confusión:
[[1847    1   29   20    0]
 [   1 1825   58   26    0]
 [ 203  215 1320   97   26]
 [  13   14   12 1852    1]
 [   0    0    0    0 1923]]


['.\\..\\Modelos\\XGB_weather_id.pkl']

RANDOM FOREST CLASSIFIER

In [12]:
# Seleccionar las características y el objetivo
features = ['year', 'month', 'day', 'precipitation', 'wind', 'humidity', 'estacion_id']
target = 'weather_id'

# Dividir los datos en características (X) y objetivo (y)
X = df[features]
y = df[target]

# Ajustar las etiquetas para que comiencen desde 0
y_adjusted = y - 1  # Ahora las clases estarán en el rango [0, 1, 2, 3, 4]

# Mostrar la distribución original de las clases
print("Distribución original de las clases:", Counter(y_adjusted))

# Crear una instancia de SMOTE
smote = SMOTE(random_state=42,k_neighbors=10)
X_resampled, y_resampled = smote.fit_resample(X, y_adjusted)

# Mostrar la nueva distribución de las clases
print("Distribución de clases después de SMOTE:", Counter(y_resampled))

# Dividir los datos balanceados en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

# Crear un modelo de Random Forest (puedes usar cualquier otro clasificador)
model = RandomForestClassifier(random_state=42, n_estimators=300, max_depth=30)

# Ajustar el modelo con los datos de entrenamiento balanceados
model.fit(X_train, y_train)

# Realizar predicciones sobre el conjunto de prueba
y_pred = model.predict(X_test)

# Reconvertir las predicciones y las etiquetas originales
y_pred = y_pred + 1  # Volver a [1, 2, 3, 4, 5]
y_test = y_test + 1  # Volver a [1, 2, 3, 4, 5]

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

# Evaluar el rendimiento del modelo
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred))

# Mostrar la matriz de confusión
print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

# Guardar el modelo entrenado usando joblib
joblib.dump(model, r'.\main\Modelos\RFC_weather_id.pkl')

Distribución original de las clases: Counter({1: 9483, 0: 9024, 2: 5959, 3: 429, 4: 105})
Distribución de clases después de SMOTE: Counter({0: 9483, 2: 9483, 1: 9483, 3: 9483, 4: 9483})
Accuracy: 0.9128967626278603

Reporte de clasificación:
              precision    recall  f1-score   support

           1       0.87      0.99      0.92      1897
           2       0.87      0.98      0.92      1910
           3       0.97      0.62      0.76      1861
           4       0.90      0.98      0.94      1892
           5       0.98      1.00      0.99      1923

    accuracy                           0.91      9483
   macro avg       0.92      0.91      0.91      9483
weighted avg       0.92      0.91      0.91      9483


Matriz de confusión:
[[1870    0    9   18    0]
 [   0 1864   17   29    0]
 [ 265  253 1155  156   32]
 [  18   19    9 1846    0]
 [   0    0    1    0 1922]]


RNN

In [4]:
# 1. Importar las bibliotecas necesarias
import pandas as pd
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

# 2. Seleccionar características y objetivo
features = ['year', 'month', 'day', 'precipitation', 'wind', 'humidity', 'estacion_id']
target = 'weather_id'

# Dividir en X (características) y y (objetivo)
X = df[features]
y = df[target]

# 3. Preprocesamiento de datos
# Normalizar las características numéricas

scaler = StandardScaler()
# Primero se normalizan los datos después de resamplear
X_resampled, y_resampled = SMOTE(random_state=42).fit_resample(X, y)

# Normalización: aplica la normalización sobre los datos resampleados
X_scaled = scaler.fit_transform(X_resampled)

# Codificar el objetivo (y) en formato "one-hot"
y_encoded = to_categorical(y_resampled)

# 4. Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=42)

# 5. Crear la arquitectura de la Red Neuronal
model = Sequential()
model.add(Dense(256, input_dim=X_train.shape[1], activation='relu'))  # Más neuronas para mayor capacidad de aprendizaje
model.add(Dropout(0.3))  # Regularización para evitar sobreajuste
model.add(Dense(128, activation='relu'))  # Segunda capa oculta
model.add(Dropout(0.3))  # Más regularización
model.add(Dense(64, activation='relu'))  # Tercera capa oculta
model.add(Dense(y_encoded.shape[1], activation='softmax'))  # Capa de salida para clasificación multiclase

# 6. Compilar el modelo
model.compile(optimizer='adam', 
              loss='categorical_crossentropy',  # Función de pérdida para clasificación multiclase
              metrics=['accuracy'])

# 7. Entrenar el modelo
history = model.fit(X_train, y_train, 
                    validation_data=(X_test, y_test), 
                    epochs=20,  # Más épocas para mejor ajuste
                    batch_size=64,  # Tamaño del lote
                    verbose=2)

# 8. Evaluar el modelo
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=2)
print(f'\nPérdida en prueba: {test_loss}')
print(f'Precisión en prueba: {test_accuracy}')

# 9. Realizar predicciones
predictions = model.predict(X_test)

# Decodificar las predicciones de vuelta a sus clases originales
predicted_classes = predictions.argmax(axis=1)

# Decodificar las etiquetas reales (y_test) si están en formato one-hot
y_test_classes = y_test.argmax(axis=1)

# 10. Calcular y mostrar métricas adicionales
cm = confusion_matrix(y_test_classes, predicted_classes)
print("\nMatriz de Confusión:")
print(cm)

print("\nReporte de Clasificación:")
print(classification_report(y_test_classes, predicted_classes))

# 11. Guardar el modelo entrenado
model.save(r'.\main\Modelos\RNN_weather_id.h5')


Epoch 1/20


c:\Users\jpetit.sta\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


593/593 - 3s - 4ms/step - accuracy: 0.7030 - loss: 0.7364 - val_accuracy: 0.7918 - val_loss: 0.5283
Epoch 2/20
593/593 - 1s - 2ms/step - accuracy: 0.7672 - loss: 0.5760 - val_accuracy: 0.8102 - val_loss: 0.4903
Epoch 3/20
593/593 - 1s - 2ms/step - accuracy: 0.7862 - loss: 0.5346 - val_accuracy: 0.8200 - val_loss: 0.4604
Epoch 4/20
593/593 - 1s - 2ms/step - accuracy: 0.7978 - loss: 0.5067 - val_accuracy: 0.8188 - val_loss: 0.4543
Epoch 5/20
593/593 - 1s - 2ms/step - accuracy: 0.8065 - loss: 0.4837 - val_accuracy: 0.8372 - val_loss: 0.4233
Epoch 6/20
593/593 - 1s - 2ms/step - accuracy: 0.8166 - loss: 0.4664 - val_accuracy: 0.8434 - val_loss: 0.4032
Epoch 7/20
593/593 - 1s - 2ms/step - accuracy: 0.8207 - loss: 0.4517 - val_accuracy: 0.8442 - val_loss: 0.4038
Epoch 8/20
593/593 - 1s - 2ms/step - accuracy: 0.8291 - loss: 0.4374 - val_accuracy: 0.8441 - val_loss: 0.3911
Epoch 9/20
593/593 - 1s - 2ms/step - accuracy: 0.8320 - loss: 0.4276 - val_accuracy: 0.8492 - val_loss: 0.3860
Epoch 10/20



[[1791    1    7   98    0]
 [  16 1800    6   88    0]
 [ 333  293  884  292   59]
 [  18   37    0 1825   12]
 [   0    0    2    0 1921]]

Reporte de Clasificación:
              precision    recall  f1-score   support

           1       0.83      0.94      0.88      1897
           2       0.84      0.94      0.89      1910
           3       0.98      0.48      0.64      1861
           4       0.79      0.96      0.87      1892
           5       0.96      1.00      0.98      1923

    accuracy                           0.87      9483
   macro avg       0.88      0.87      0.85      9483
weighted avg       0.88      0.87      0.85      9483

